In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import switch_cwd_to_root

switch_cwd_to_root()

import scanpy as sc

In [ ]:
path = "data/xenium/processed/04-kidney_tcr_tsub.h5ad"
adata = sc.read_h5ad(path)
adata.X = adata.layers["counts"].copy()

tcell_keys = ["CD4+", "TFH", "Tregs", "MAIT", "CD8+", "NK/NKT", "gdT"]

assert all(tcell_key in adata.obs["tcell_subtype"].unique() for tcell_key in tcell_keys)

# define broad and fine cell types
adata.obs["cell_type_l1"] = adata.obs["cell_type_no_tcr"].astype(str)
t_mask = adata.obs["cell_type_l1"] == "T"
adata.obs.loc[t_mask, "cell_type_l1"] = adata.obs.loc[t_mask, "tcell_subtype"]
adata.obs.loc[adata.obs["cell_type_l1"].isin(tcell_keys), "cell_type_l1"] = "T"


adata.obs["cell_type_l2"] = adata.obs["tcell_subtype"].copy()

# merge TFH with CD4+
mapping = {
    "TFH": "CD4+",
    "NK/NKT": "NKT-like",
}
adata.obs["cell_type_l2"] = (
    adata.obs["cell_type_l2"].astype(str).replace(mapping).astype("category")
)


adata

AnnData object with n_obs × n_vars = 510139 × 436
    obs: 'sample', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'slide', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'n_counts', 'condition', 'cc', 'cell_type_no_tcr', 'cell_type_no_tcr_prob', 'tcell_subtype', 'is_ATL', 'is_B', 'is_CNT', 'is_DCT', 'is_DTL', 'is_EC', 'is_FIB', 'is_IC', 'is_MAST', 'is_MC', 'is_MDC', 'is_Mac', 'is_N', 'is_NEU', 'is_PC', 'is_PEC', 'is_PL', 'is_POD', 'is_PT', 'is_PapE', 'is_T', 'is_TAL', 'is_VSM/P', 'is_cDC', 'is_cycMNP', 'is_glom. EC', 'is_pDC', 'leiden', 'leiden_0', 'leiden_1', 'leiden_10', 'leiden_11', 'leiden_12', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'leiden_6', 'leiden_7', 'leiden_8', 'leide

In [6]:
label_key = "cell_type_l2"
# label_key = "cell_type_l1"


adata.obs[label_key].value_counts()

cell_type_l2
PT          120985
FIB          79706
Mac          51415
TAL          46477
EC           41864
MDC          21738
PC           19094
IC           18190
VSM/P        16335
CNT          14542
DCT          14132
CD4+          9258
POD           8584
glom. EC      6554
CD8+          6509
PL            6156
B             5862
DTL           5230
PEC           4937
MC            2816
unknown       2241
NKT-like      1842
Tregs         1158
MAST          1154
MAIT           820
cycMNP         771
N              544
gdT            446
ATL            396
cDC            303
pDC             37
PapE            27
NEU             16
Name: count, dtype: int64

In [3]:
label_key = "cell_type_l2"
# label_key = "cell_type_l1"


adata.obs[label_key].value_counts()

cell_type_l2
PT          120985
FIB          79706
Mac          51415
TAL          46477
EC           41864
MDC          21738
PC           19094
IC           18190
VSM/P        16335
CNT          14542
DCT          14132
CD4+          9258
POD           8584
glom. EC      6554
CD8+          6509
PL            6156
B             5862
DTL           5230
PEC           4937
MC            2816
unknown       2241
NKT-like      1842
Tregs         1158
MAST          1154
MAIT           820
cycMNP         771
N              544
gdT            446
ATL            396
cDC            303
pDC             37
PapE            27
NEU             16
Name: count, dtype: int64

In [4]:
adata.write_h5ad("data/xenium/processed/04.1-kidney_tcr_tsub_harmonized.h5ad")